# 🫀 실험 16 — **유도 4구성 확장**: V1 한 개가 전벽을 얼마나 되찾는가

**MedKOS / `notebooks/exp16_four_lead_configs.ipynb`** · 퀘스트 `ailab-2026-0015`
**다운로드 없음**(실험15 전량 캐시 재사용) · 4구성 × 3시드 × 5겹 = **60회 (~60분)**

---

## 이 실험이 답하는 것

지금까지는 구성이 **둘**이었다 — `{I,II}`(사지 2개)와 `{12}`(전부). 그래서
"저유도는 얼마나 손해인가"까지만 말할 수 있었고, **"어떤 유도를 하나 더 붙여야
하는가"** 는 말할 수 없었다. 웨어러블 설계에서 실제로 필요한 답은 후자다.

구성을 넷으로 늘리면 **유도 수가 같은 두 구성**을 맞붙일 수 있다:

| 구성 | 유도 | 평면 | 실제 기기 |
|---|---|---|---|
| `{II}` | 1개 | 전두면 | 단일유도 패치·스마트워치 (Lead I 급) |
| `{I,II}` | **2개** | **전두면 전체** | 3전극 홀터 (aVR·aVL·aVF 가 계산으로 따라온다) |
| `{II,V1}` | **2개** | **전두면 1 + 횡단면 1** | 흉부 패치 1개를 더 붙인 형태 |
| `{12}` | 12개 | 전부 | 표준 심전도 |

`{I,II}` 와 `{II,V1}` 은 **전극 수가 같다.** 다른 것은 두 번째 전극을 **어느 평면에
놓느냐** 뿐이다. 이 둘이 소견에 따라 **뒤바뀌면**, 그것이 차별점 A 의 가장 강한 형태이자
**곧바로 전극 배치 권고**가 된다.

## 왜 V1 인가 — 순환기 수업으로 돌아가서

하벽경색(IMI)은 **II·III·aVF** 의 Q파로 읽는다. 셋 다 사지유도다.
그래서 `{I,II}` 만 있어도 아인트호벤·골드버거로 **전두면 6유도가 전부 복원**된다
(`III = II − I`, `aVL = I − II/2` …). 하벽은 사지유도로 충분할 것이다.

전중격경색(ASMI)은 **V1–V3** 의 Q파·R파 감소로 읽는다. 이건 **가슴 앞쪽**을 보는
유도라 사지유도 조합으로는 **원리적으로 만들 수 없다**. 전두면 벡터에 없는 정보다.

> **그러면 V1 하나만 붙이면 얼마나 되찾는가?** 그게 이 실험의 핵심 숫자다.

## 무엇을 얼마나 돌리나 — **전부 새로 학습한다**

실험15 의 `{I,II}`·`{12}` arm 이 있지만 **재사용하지 않는다.**
실험15d 의 G0 에서 **같은 시드로 재학습해도 저장 arm 이 재현되지 않았다**(GPU 비결정성).
이번 헤드라인은 `{I,II}` 대 `{II,V1}` 의 **교차**인데, 한쪽 팔만 다른 날·다른 세션에서
학습된 값이면 **환경 드리프트가 교차 위에 그대로 얹힌다.** 그래서 4구성을 같은 세션에서
같은 조건으로 돌린다.

대신 실험15 의 arm 을 **드리프트 대조군**으로 읽어 그 크기를 숫자로 남긴다(G0).
실험15d 가 "재현 안 됨"만 말하고 **얼마나**를 못 남긴 자리를 여기서 메운다.

| | 학습 |
|---|---|
| 4구성 × 3시드 × 5겹 | **60회 (~60분)** |

겹마다 `save_arm` 으로 체크포인트하므로 **런타임이 끊겨도 이어서 돌린다.**

## 사전등록 (결과 보기 전에 고정)

지표는 **AUROC 기반 잔여 결손** `D(c) = AUROC({12}) − AUROC(c)` 로 통일한다.
AUPRC 는 유병률에 묶여 부위 간 비교가 안 되지만(실험15 에서 확인), AUROC 는 유병률
무관이라 **부위·구성을 가로질러 더할 수 있다.** AUPRC·동작점은 함께 보고만 한다.

**판정은 실험15d 가 정한 새 규약대로 `시드 수준 t-CI(df=2)`** 로 한다.
따로 학습한 두 모델을 비교하므로 테스트셋 부트스트랩만으로는 부족하다.

| | 예측 | 무엇이 걸려 있나 |
|---|---|---|
| **G0** | 유도 순서를 항등식으로 확인 + 실험15 대비 드리프트 보고 | 순서가 틀리면 조용히 다른 실험이 된다 |
| **P-1** | **단조성**: 모든 부위에서 `D({II}) ≥ D({I,II})` 이고 `D({II}) ≥ D({II,V1})` (7부위 중 ≥6) | 유도를 더했는데 나빠지면 계측기 고장이다(실험10-full 이 그렇게 무효가 됐다) |
| **P-2** | **전두면 포화**: `IMI` 의 `D({I,II})` t-CI 상한 < **0.02** | 하벽경색은 사지 2개로 12유도를 따라잡는가 |
| **P-3 ★** | **V1 의 선택성**: `G(s) = D({II},s) − D({II,V1},s)` 에서 **횡단면 평균 − 전두면 평균 > 0** | 유도 **하나**를 개입으로 쓴 평면 가설의 직접 검정 |
| **P-4 ★★** | **교차**: `[AUROC({II,V1}) − AUROC({I,II})]` 가 횡단면에서 전두면보다 크다(차 > 0) | 전극 수가 같을 때 **어디에 붙일지**가 소견에 따라 뒤바뀌는가 |

**붕괴 감시**는 실험15 와 같다 — 부위별로 AUPRC < 유병률×1.2 면 그 부위 제외,
과반이 죽거나 대조 구성 한쪽이 통째로 죽으면 판정 무효.

## 이 실험이 무엇을 정하나

- `P-4` ✅ → **배치 권고가 나온다**: 전벽을 보려면 사지 전극을 하나 더 붙이지 말고
  **가슴에 하나 붙여라**. 차별점 A 가 "설명"에서 "설계 규칙"으로 승격된다.
- `P-3` ✅ + `P-2` ✅ → **소견별 최소 유도 세트**를 표로 낼 수 있다(2단계 라벨 트리의 입력).
- `P-1` ❌ → 계측기 이상. 먼저 고치고 나머지 판정은 보류한다.


In [ ]:
# CELL 0 — 공용 사전점검 (pipelines/ecg_preflight.py 인라인)
class LabelVocabError(ValueError):
    pass

# ── pipelines/ecg_preflight.py 인라인 (원본·테스트는 repo)
def assert_label_vocab(requested, available, kind="label", counts=None, min_count=1):
    """요청한 이름이 실제 어휘에 **전부** 있는지 확인한다. 하나라도 없으면 예외.

    0건은 "데이터에 그 소견이 없다"가 아니라 **대개 이름을 잘못 골랐다**는 뜻이다.
    그 둘을 구별하려고 어휘 자체를 대조한다.

    requested : 쓰려는 이름들
    available : 데이터에서 실제로 관측된 이름 집합
    counts    : {이름: 건수} (있으면 min_count 미만도 함께 보고)
    """
    requested, available = list(requested), set(available)
    unknown = [r for r in requested if r not in available]
    if unknown:
        raise LabelVocabError(
            f"{kind} 어휘에 없는 이름 {unknown}.\n"
            f"  → 0건이 나온 이유는 '데이터에 없어서'가 아니라 **이름이 틀려서**다.\n"
            f"  실제 어휘({len(available)}개): {sorted(available)}"
        )
    thin = []
    if counts:
        thin = [(r, counts.get(r, 0)) for r in requested if counts.get(r, 0) < min_count]
    return {"ok": True, "n_requested": len(requested), "thin": thin}

def decide(lo, hi, thr, direction):
    """사전등록 관문의 **유일한** 계약: 지지(True) / 기각(False) / 미결(None).

    CI 가 임계값을 걸치면 **기각이 아니라 미결**이다. 검정력 부족을 반증으로
    위장하지 않기 위해서다. 점추정 2분 채점은 금지한다(실험13b·14 에서 그 실수를 했다).
    """
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr:
            return True
        if hi < thr:
            return False
    else:
        if hi < thr:
            return True
        if lo > thr:
            return False
    return None

MARK = {True: "✅ 지지", False: "❌ 기각", None: "⚠️ 미결"}

def collapse_report(scores, floors, names, lift=1.2):
    """붕괴 감시 — 단위(클래스·부위)별로 판정한다.

    scores : {이름: 성능}   floors : {이름: 무작위 수준(유병률 등)}
    성능이 floor 의 `lift` 배도 안 되면 그 단위는 죽은 것으로 본다.

    ★ 하나 죽었다고 실험 전체를 무효화하지 않는다. 무효 판단은 호출자가
      '과반이 죽었는가 / 대조군 한쪽이 통째로 죽었는가'로 따로 한다.
    """
    dead = [n for n in names if scores.get(n, 0.0) < floors.get(n, 0.0) * lift]
    alive = [n for n in names if n not in dead]
    return {"dead": dead, "alive": alive,
            "fatal_majority": len(dead) >= len(names) / 2 if names else True}

def assert_arm_shape(arm, expected_rows, name="arm"):
    """저장된 arm 의 행 수가 **겹 크기**인지 확인한다.

    MedKOSRun.save_arm 은 그 겹의 예측만 저장한다(전체 길이가 아니다).
    겹 순서는 `np.where(CV == k)[0]` 의 오름차순이므로,
      OOF[np.where(CV == k)[0]] = load_arm(...)      ← 이렇게 **넣는다**
      load_arm(...)[전역인덱스]                       ← 이렇게 자르면 IndexError
    실험15d G0 에서 이 혼동으로 터졌다.
    """
    n = arm.shape[0]
    if n != expected_rows:
        raise ValueError(
            f"{name} 행 수 {n} != 기대 {expected_rows}.\n"
            "  → arm 은 **겹 크기**로 저장된다. 전역 인덱스로 자르지 말고 "
            "OOF[np.where(CV==k)[0]] = arm 형태로 넣을 것."
        )
    return {"ok": True, "rows": n}

FRONTAL_IDENTITIES = (
    ("III", 2, lambda I, II: II - I),                 # 아인트호벤
    ("aVR", 3, lambda I, II: -(I + II) / 2.0),        # 골드버거
    ("aVL", 4, lambda I, II: I - II / 2.0),
    ("aVF", 5, lambda I, II: II - I / 2.0),
)

def assert_lead_order(X, tol=0.02, sample=200, seed=0):
    """12유도 캐시의 **채널 순서**를 신호 자체로 검증한다.

    헤더의 유도 이름을 믿지 말고 아인트호벤·골드버거 항등식으로 확인한다:
        III = II − I,  aVR = −(I+II)/2,  aVL = I − II/2,  aVF = II − I/2
    넷이 모두 맞으면 0..5 = I,II,III,aVR,aVL,aVF 이고 표준 순서상 6..11 = V1..V6 이다.
    → `{I,II}` = [0,1], `{II,V1}` = [1,6] 을 쓸 근거가 생긴다.

    유도 순서를 틀리면 **예외 없이 조용히 다른 실험**이 된다. 그래서 잰다.
    ※ 원신호(mV) 전제 — 채널별로 정규화한 배열에는 쓸 수 없다.
    """
    import numpy as np
    if X.ndim != 3 or X.shape[2] != 12:
        raise ValueError(f"X 는 (n, t, 12) 여야 한다 — 받은 모양 {X.shape}")
    rs = np.random.RandomState(seed)
    idx = rs.choice(len(X), size=min(sample, len(X)), replace=False)
    S = X[idx].astype("float64")
    I, II = S[:, :, 0], S[:, :, 1]
    report, bad = {}, []
    for name, j, f in FRONTAL_IDENTITIES:
        want = f(I, II)
        scale = np.abs(want).mean() + 1e-9
        err = float(np.abs(S[:, :, j] - want).mean() / scale)
        report[name] = err
        if err > tol:
            bad.append(f"{name}(ch{j}) 상대오차 {err:.3f}")
    if bad:
        raise ValueError(
            "유도 순서가 표준(I,II,III,aVR,aVL,aVF,V1..V6)이 아니다: " + ", ".join(bad) + "\n"
            "  → 항등식이 깨졌다는 것은 채널 배치가 다르거나 채널별 정규화가 걸렸다는 뜻이다.\n"
            "  마스크 인덱스([0,1] 사지 · [1,6] II+V1)를 그대로 쓰면 조용히 다른 실험이 된다."
        )
    return {"ok": True, "n_checked": len(idx), "rel_err": report}

def boot_indices(n, B, seed):
    """부트스트랩 인덱스를 **생성기로** 돌려준다.

    미리 리스트로 만들면 B=4000, n=16k 에서 520MB 다. 같은 시드로 매번 다시 돌리면
    메모리 0 이면서 **여러 군이 같은 재표본 축을 공유**한다(짝지은 비교의 전제).
    """
    import numpy as np
    rs = np.random.RandomState(seed)
    for _ in range(B):
        yield rs.randint(0, n, n)

print("사전점검 적재: assert_label_vocab · decide · collapse_report · "
      "assert_arm_shape · assert_lead_order · boot_indices")

In [ ]:
# CELL 1 — 설정 + 실험15 산출물 연결(드리프트 대조용)
!pip -q install wfdb

import os, sys, json, time, ast, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ★★ 실험15·15c·15d 와 한 글자도 달라선 안 되는 블록 (백본·분할·시드 공식)
K_FOLD, EPOCHS, SEED0, BOOT, NMIN = 5, 20, 20260801, 2000, 50
SITE_CANDIDATES = ["IMI", "ILMI", "IPMI", "IPLMI", "ASMI", "AMI", "ALMI", "LMI", "PMI"]
SITE_PLANE = {"IMI": "전두면", "ILMI": "혼합", "IPMI": "혼합", "IPLMI": "혼합",
              "ASMI": "횡단면", "AMI": "횡단면", "ALMI": "혼합",
              "LMI": "혼합", "PMI": "횡단면"}
# ★★ 여기까지

# ── 이번에 바꾸는 단 하나: 구성 2개 → 4개
#    채널 인덱스는 CELL 2 에서 아인트호벤 항등식으로 검증한 뒤에만 유효하다.
CONFIGS = {"II":    [1],            # 단일유도 — 스마트워치급
           "I+II":  [0, 1],         # 사지 2개 = 전두면 전체(계산으로 6유도 복원)
           "II+V1": [1, 6],         # 같은 2개인데 하나를 횡단면으로
           "12":    list(range(12))}
SEEDS   = [0, 1, 2]                 # 실험15d 규약 — 결론용 비교는 최소 3시드
REF     = "12"                      # 잔여 결손 D 의 기준 구성
SAT_THR = 0.02                      # P-2 포화 문턱(AUROC 단위)

CONFIG = dict(exp="exp16_four_lead_configs", quest="ailab-2026-0015",
              parent_exp=["exp15_mi_loc_head", "exp15d_seed_var"],
              purpose="유도 구성을 4개로 늘려 '어느 평면에 전극을 하나 더 붙일 것인가'에 답한다",
              change_one_thing="실험15 와 백본·분할·에폭·손실 동일. 구성 2개 → 4개, 시드 1개 → 3개",
              no_reuse=("실험15 arm 을 재사용하지 않는다 — 15d G0 에서 환경 드리프트가 확인됐고 "
                        "헤드라인이 두 구성의 교차라 한쪽 팔만 다른 세션이면 드리프트가 얹힌다"),
              metric="D(c) = AUROC({12}) - AUROC(c)  · 유병률 무관이라 부위 간 합산 가능",
              judged_on="시드 수준 t-CI(df=2) — 실험15d 가 정한 규약",
              configs={k: v for k, v in CONFIGS.items()}, seeds=SEEDS,
              sites=SITE_CANDIDATES, site_plane=SITE_PLANE,
              predictions={
                  "G0": "유도 순서 항등식 검증 + 실험15 대비 드리프트 크기 보고",
                  "P-1": "단조성 D({II}) >= D({I+II}) 이고 D({II}) >= D({II+V1}) — 7부위 중 6 이상",
                  "P-2": f"IMI 의 D({{I+II}}) t-CI 상한 < {SAT_THR} (전두면 포화)",
                  "P-3": "V1 이득 G = D({II}) - D({II+V1}) 이 횡단면 > 전두면",
                  "P-4": "AUROC({II+V1}) - AUROC({I+II}) 가 횡단면 > 전두면 (교차)"},
              collapse="부위별 AUPRC < 유병률 x 1.2 면 그 부위 제외. 과반 사망 시 판정 무효",
              k_fold=K_FOLD, epochs=EPOCHS, seed0=SEED0, boot=BOOT, nmin=NMIN)
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm, subprocess
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp16_four_cfg", CONFIG, project=PROJECT)

# ── 실험15 를 찾는다(드리프트 대조용 — 없어도 실험은 돈다)
REG = os.path.join(PROJECT, "registry.jsonl")
P15 = None
for line in (open(REG) if os.path.exists(REG) else []):
    try:
        r = json.loads(line)
    except Exception:
        continue
    if r.get("exp_id") == "exp15_mi_loc_head" and os.path.isdir(r.get("dir", "")):
        P15 = r["dir"]
run.log(f"실험15 산출물: {P15 or '없음 — G0 드리프트 대조는 생략된다'}")
SITES15 = (json.load(open(os.path.join(P15, "result.json"), encoding="utf-8"))["sites"]
           if P15 else None)

def arm_at(d, name):
    p = os.path.join(d, "arms", name, "probs.npy")
    return np.load(p) if (d and os.path.exists(p)) else None

In [ ]:
# CELL 2 — 캐시 재사용 + 라벨 + 【G0-a】 유도 순서 검증 (다운로드 없음)
import pandas as pd, subprocess

PTB = "/content/ptbxl"
os.makedirs(PTB, exist_ok=True)
BASE = "https://physionet.org/files/ptb-xl/1.0.3"
d = os.path.join(PTB, "ptbxl_database.csv")
if not (os.path.exists(d) and os.path.getsize(d) > 0):
    subprocess.run(["wget", "-q", "-O", d, f"{BASE}/ptbxl_database.csv"])
df = pd.read_csv(d, index_col="ecg_id")

CACHE = run.data("ptbxl_12lead_all.npz")
if not os.path.exists(CACHE):
    raise RuntimeError(f"전량 캐시가 없습니다: {CACHE} — 실험15 CELL 2 를 먼저 돌리세요")
z = np.load(CACHE, allow_pickle=True)
X, FOLD10, EID = z["X"], z["fold"], z["eid"]
CV = (FOLD10 - 1) % K_FOLD

# ── 【G0-a】 채널 순서를 **신호로** 확인한다.
#    이름표를 믿지 않는다. III = II − I 같은 항등식이 맞아야 [0,1]=I,II 이고 [6]=V1 이다.
#    여기서 틀리면 {II,V1} 이 조용히 다른 유도가 되어 실험 전체가 무의미해진다.
lead_chk = assert_lead_order(X)
run.log("【G0-a】 유도 순서 검증 통과 — 상대오차 "
        + " · ".join(f"{k} {v:.4f}" for k, v in lead_chk["rel_err"].items()))
run.log(f"  → [0,1] = I,II · [6] = V1 확정 (표본 {lead_chk['n_checked']}건)")

dfa = df.loc[EID]
dfa["codes"] = dfa.scp_codes.apply(lambda s: sorted(ast.literal_eval(s).keys()))
vocab = {c for cs in dfa.codes for c in cs}
counts = {s: int(sum(s in cs for cs in dfa.codes)) for s in SITE_CANDIDATES}
# scp **코드** 층위다(subclass 아님) — 실험13·13b·15 가 여기서 틀렸었다
assert_label_vocab(SITE_CANDIDATES, vocab, kind="scp 코드", counts=counts, min_count=NMIN)
SITES = [s for s in SITE_CANDIDATES if counts[s] >= NMIN]
if SITES15 is not None and SITES != SITES15:
    raise RuntimeError(f"부위 목록이 실험15와 다릅니다: {SITES} vs {SITES15}")
NS = len(SITES)
Ymul = np.stack([[s in c for s in SITES] for c in dfa.codes]).astype("float32")

run.log(f"\n캐시 X{X.shape} · 부위 {NS}개")
run.log(f"  {'부위':<8}{'평면':<7}{'n':>8}{'유병률':>9}")
for j, s in enumerate(SITES):
    run.log(f"  {s:<8}{SITE_PLANE[s]:<7}{int(Ymul[:, j].sum()):>8,}{Ymul[:, j].mean():>9.4f}")
FRONT = [s for s in SITES if SITE_PLANE[s] == "전두면"]
TRANS = [s for s in SITES if SITE_PLANE[s] == "횡단면"]
run.log(f"\n평면 사전배정(실험13b·15 와 동일): 전두면 {FRONT} · 횡단면 {TRANS}")
if not FRONT or not TRANS:
    raise RuntimeError("전두면·횡단면 중 한쪽이 비었습니다 — P-3·P-4 를 채점할 수 없습니다")

MASKS = {}
for c, idx in CONFIGS.items():
    m = np.zeros(12, "float32"); m[idx] = 1.0
    MASKS[c] = m
run.log("마스크: " + " · ".join(f"{c}={int(v.sum())}유도" for c, v in MASKS.items()))

In [ ]:
# CELL 3 — 학습 4구성 × 3시드 × 5겹 = 60회 (겹마다 체크포인트 → 끊겨도 이어서)
import tensorflow as tf
from tensorflow.keras import layers, models

def build_head(seed):
    """★ 실험15·15c·15d 와 완전히 동일한 백본·헤드·손실."""
    tf.keras.utils.set_random_seed(seed)
    si = layers.Input((X.shape[1], 12))
    x = si
    for f, k in ((32, 9), (64, 7), (128, 5), (128, 3)):
        x = layers.Conv1D(f, k, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling1D(2)(x)
    h = layers.Dense(64, activation="relu")(layers.GlobalAveragePooling1D()(x))
    h = layers.Dropout(0.3)(h); h = layers.Dense(64, activation="relu")(h)
    m = models.Model(si, layers.Dense(NS, activation="sigmoid")(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss="binary_crossentropy")     # 가중 없음 — pos_weight 는 15b 담당
    return m

def split(k):
    """★ 실험15 와 동일한 분할. 시드에 무관하게 겹만으로 결정된다."""
    te = np.where(CV == k)[0]; rest = np.where(CV != k)[0]
    rs_ = np.random.RandomState(SEED0 + k); rs_.shuffle(rest)
    n_val = max(int(len(rest) * 0.12), 200)
    return te, rest[:n_val], rest[n_val:]

# OOF[구성][시드] = (전체 길이 × 부위) 배열
OOF = {c: {s: np.zeros((len(EID), NS), "float32") for s in SEEDS} for c in CONFIGS}
t0, done, total = time.time(), 0, len(CONFIGS) * len(SEEDS) * K_FOLD
run.log(f"총 {total}회 학습 예정")
for c in CONFIGS:
    mk = MASKS[c]
    for sd in SEEDS:
        for k in range(K_FOLD):
            te = np.where(CV == k)[0]
            a = run.load_arm(f"{c}_s{sd}_f{k}")
            if a is None:
                te, va, tr = split(k)
                m = build_head(SEED0 + 100 * k + 15 + sd)   # ★ 15·15c·15d 와 같은 공식
                m.fit(X[tr] * mk, Ymul[tr], validation_data=(X[va] * mk, Ymul[va]),
                      epochs=EPOCHS, batch_size=128, verbose=0)
                a = m.predict(X[te] * mk, batch_size=512, verbose=0)
                run.save_arm(f"{c}_s{sd}_f{k}", a)
                if k == 0 and sd == 0:
                    run.save_model(m, f"head_{c}")
                tf.keras.backend.clear_session()
            # arm 은 **겹 크기**로 저장된다 — 전역 인덱스로 자르지 말고 넣는다(15d 의 교훈)
            assert_arm_shape(a, len(te), name=f"{c}_s{sd}_f{k}")
            OOF[c][sd][te] = a
            done += 1
            if done == 1:
                per = time.time() - t0
                run.log(f"  ⏱ 첫 학습 {per:.0f}s → 전체 {total}회 예상 **{per*total/60:.0f}분**")
        run.log(f"  {c:<6} 시드{sd} 완료 ({done}/{total} · {time.time()-t0:.0f}s)")
run.log(f"\n총 {time.time()-t0:.0f}s")

In [ ]:
# CELL 4 — 【G0-b】 드리프트 대조 + 구성×부위×시드 지표
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy import stats

def spec_at_sens(score, pos, target=0.90):
    p, n = score[pos], score[~pos]
    if len(p) == 0 or len(n) == 0:
        return np.nan, np.nan, np.nan
    thr = float(np.quantile(p, 1.0 - target, method="lower"))
    return float((p >= thr).mean()), float((n < thr).mean()), float((score >= thr).mean())

def t_ci(vals, conf=0.95):
    """시드 수준 t-CI(df = n−1). 실험15d 가 정한 판정 기준."""
    v = np.asarray(vals, float); n = len(v)
    m = float(v.mean())
    if n < 2:
        return m, np.nan, np.nan, 0.0
    sd = float(v.std(ddof=1))
    h = float(stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n))
    return m, m - h, m + h, sd

# ── 【G0-b】 실험15 arm 과의 환경 드리프트. 15d 가 "재현 안 됨"만 말하고 크기를 안 남겼다.
run.log("\n" + "=" * 100)
run.log("【G0-b】 환경 드리프트 — 실험15 저장 arm vs 이번 세션 시드0 (같은 시드 공식)")
run.log("=" * 100)
drift = {}
if P15:
    for c15, c16 in (("I+II", "I+II"), ("12", "12")):
        old = np.zeros((len(EID), NS), "float32"); ok = True
        for k in range(K_FOLD):
            a = arm_at(P15, f"{c15}_f{k}")
            if a is None:
                ok = False; break
            old[np.where(CV == k)[0]] = a
        if not ok:
            run.log(f"  {c16}: 실험15 arm 없음 — 생략"); continue
        for j, s in enumerate(SITES):
            y = Ymul[:, j].astype(bool)
            d_ = roc_auc_score(y, OOF[c16][0][:, j]) - roc_auc_score(y, old[:, j])
            drift[f"{c16}|{s}"] = float(d_)
        v = [drift[f"{c16}|{s}"] for s in SITES]
        run.log(f"  {c16:<6} ΔAUROC(이번−실험15) 평균 {np.mean(v):+.4f} · "
                f"최대 |Δ| {np.abs(v).max():.4f} · 부위별 "
                + " ".join(f"{s}{drift[f'{c16}|{s}']:+.3f}" for s in SITES))
    mx = max(abs(x) for x in drift.values()) if drift else 0.0
    run.log(f"\n  → 최대 드리프트 |Δ| = {mx:.4f}")
    run.log("  " + ("⚠️ 0.02 초과 — 실험15 수치와의 **직접 비교**에 단서를 단다. "
                    "이번 실험 내부 비교는 같은 세션이라 영향 없다." if mx > 0.02 else
                    "✅ 0.02 이하 — 실험15 수치와 이어 붙여도 된다."))
else:
    run.log("  실험15 산출물이 없어 생략")

# ── 구성 × 부위 × 시드 지표
run.log("\n" + "=" * 118)
run.log("【구성별 성능】 시드 3개 평균 · D = AUROC(12) − AUROC(구성)")
run.log("=" * 118)
M = {}       # M[구성][부위] = {지표: [시드별 값]}
for c in CONFIGS:
    M[c] = {}
    for j, s in enumerate(SITES):
        y = Ymul[:, j].astype(bool)
        M[c][s] = {"auprc": [], "auroc": [], "spec": [], "alarm": []}
        for sd in SEEDS:
            sc = OOF[c][sd][:, j]
            M[c][s]["auprc"].append(float(average_precision_score(y, sc)))
            M[c][s]["auroc"].append(float(roc_auc_score(y, sc)))
            _, sp, al = spec_at_sens(sc, y)
            M[c][s]["spec"].append(sp); M[c][s]["alarm"].append(al)

D = {c: {s: [M["12"][s]["auroc"][i] - M[c][s]["auroc"][i] for i in range(len(SEEDS))]
         for s in SITES} for c in CONFIGS}

run.log(f"  {'부위':<7}{'평면':<7}{'n':>6}"
        + "".join(f"{'AUROC ' + c:>15}" for c in CONFIGS)
        + f"{'D(II)':>9}{'D(I+II)':>10}{'D(II+V1)':>11}")
for j, s in enumerate(SITES):
    run.log(f"  {s:<7}{SITE_PLANE[s]:<7}{int(Ymul[:, j].sum()):>6,}"
            + "".join(f"{np.mean(M[c][s]['auroc']):>15.4f}" for c in CONFIGS)
            + f"{np.mean(D['II'][s]):>+9.4f}{np.mean(D['I+II'][s]):>+10.4f}"
            + f"{np.mean(D['II+V1'][s]):>+11.4f}")

# ── 붕괴 감시(부위별 — 실험15 와 동일 계약)
run.log("\n【붕괴 감시】 AUPRC < 유병률 × 1.2 = 사실상 무작위")
prev = {s: float(Ymul[:, j].mean()) for j, s in enumerate(SITES)}
dead_any, rep = set(), {}
for c in CONFIGS:
    r = collapse_report({s: np.mean(M[c][s]["auprc"]) for s in SITES}, prev, SITES)
    rep[c] = r
    dead_any |= set(r["dead"])
    run.log(f"  {c:<6} 사망 {r['dead'] or '없음'}"
            + ("  ⛔ 과반 사망" if r["fatal_majority"] else ""))
FATAL = any(rep[c]["fatal_majority"] for c in CONFIGS)
ALIVE = [s for s in SITES if s not in dead_any]
run.log(f"  → 전 구성에서 살아남은 부위 {ALIVE}")
if FATAL:
    run.log("  ⛔ 과반이 죽었다 — 아래 판정은 무효로 읽는다")

# ── 동작점(민감도 0.90) — 배포에 쓰는 한 점
run.log("\n【민감도 0.90 동작점】 특이도 / 경보율 (시드 평균)")
run.log(f"  {'부위':<7}" + "".join(f"{c:>18}" for c in CONFIGS))
for s in SITES:
    run.log(f"  {s:<7}" + "".join(
        f"{np.nanmean(M[c][s]['spec']):>9.3f}/{np.nanmean(M[c][s]['alarm']):>8.3f}"
        for c in CONFIGS))

In [ ]:
# CELL 5 — 사전등록 채점 (판정은 시드 수준 t-CI — 실험15d 규약)
run.log("\n" + "=" * 100)
run.log("【사전등록 채점】  판정 기준 = 시드 수준 t-CI(df=2)")
run.log("=" * 100)
V = {}

# ── P-1 단조성: 유도를 더하면 결손이 줄어드는가 (계측기 확인)
mono, viol = 0, []
for s in SITES:
    a, b, c_ = np.mean(D["II"][s]), np.mean(D["I+II"][s]), np.mean(D["II+V1"][s])
    okk = (a >= b) and (a >= c_)
    mono += okk
    if not okk:
        viol.append(f"{s}(D:II {a:+.3f} · I+II {b:+.3f} · II+V1 {c_:+.3f})")
V["P-1"] = bool(mono >= len(SITES) - 1)
run.log(f"  P-1 단조성 → {MARK[V['P-1']]}  {mono}/{len(SITES)} 부위에서 성립")
for v in viol:
    run.log(f"      ⚠️ 위반 {v}")
if viol:
    run.log("      ※ 유도를 더했는데 나빠졌다 = 용량·최적화 문제. 정보론적으로는 불가능하다")

# ── P-2 전두면 포화: 하벽은 사지 2개로 12유도를 따라잡는가
run.log("")
for s in FRONT:
    m, lo, hi, sd = t_ci(D["I+II"][s])
    v = decide(lo, hi, SAT_THR, "<")
    if s == FRONT[0]:
        V["P-2"] = v
    run.log(f"  P-2 {s} 전두면 포화 D(I+II) = {m:+.4f} [{lo:+.4f}, {hi:+.4f}] "
            f"(시드SD {sd:.4f}) vs 문턱 {SAT_THR} → {MARK[v]}")
    run.log(f"      시드별: " + " ".join(f"{x:+.4f}" for x in D["I+II"][s]))

# ── P-3 ★ V1 의 선택성: 유도 하나를 개입으로 쓴 평면 검정
run.log("")
G = {s: [D["II"][s][i] - D["II+V1"][s][i] for i in range(len(SEEDS))] for s in SITES}
run.log("  【V1 이득 G = D(II) − D(II+V1)】 V1 하나가 되찾는 AUROC")
for s in SITES:
    m, lo, hi, _ = t_ci(G[s])
    run.log(f"      {s:<7}{SITE_PLANE[s]:<7}{m:>+9.4f} [{lo:+.4f}, {hi:+.4f}]"
            + ("   ← 횡단면" if s in TRANS else "   ← 전두면" if s in FRONT else ""))
gt = [np.mean([G[s][i] for s in TRANS]) for i in range(len(SEEDS))]
gf = [np.mean([G[s][i] for s in FRONT]) for i in range(len(SEEDS))]
diff3 = [gt[i] - gf[i] for i in range(len(SEEDS))]
m3, lo3, hi3, sd3 = t_ci(diff3)
V["P-3"] = decide(lo3, hi3, 0.0, ">")
run.log(f"\n  P-3 V1 선택성 (횡단면 {np.mean(gt):+.4f} − 전두면 {np.mean(gf):+.4f}) "
        f"= {m3:+.4f} [{lo3:+.4f}, {hi3:+.4f}] → {MARK[V['P-3']]}")
run.log(f"      시드별: " + " ".join(f"{x:+.4f}" for x in diff3) + f"  (시드SD {sd3:.4f})")

# ── P-4 ★★ 교차: 전극 수가 같을 때 어디에 붙일 것인가
run.log("")
run.log("  【교차 W = AUROC(II+V1) − AUROC(I+II)】 전극 2개, 배치만 다르다")
W = {s: [M["II+V1"][s]["auroc"][i] - M["I+II"][s]["auroc"][i] for i in range(len(SEEDS))]
     for s in SITES}
for s in SITES:
    m, lo, hi, _ = t_ci(W[s])
    win = "II+V1(가슴)" if m > 0 else "I+II(사지)"
    run.log(f"      {s:<7}{SITE_PLANE[s]:<7}{m:>+9.4f} [{lo:+.4f}, {hi:+.4f}]   우세 {win}")
wt = [np.mean([W[s][i] for s in TRANS]) for i in range(len(SEEDS))]
wf = [np.mean([W[s][i] for s in FRONT]) for i in range(len(SEEDS))]
diff4 = [wt[i] - wf[i] for i in range(len(SEEDS))]
m4, lo4, hi4, sd4 = t_ci(diff4)
V["P-4"] = decide(lo4, hi4, 0.0, ">")
run.log(f"\n  P-4 교차 (횡단면 {np.mean(wt):+.4f} − 전두면 {np.mean(wf):+.4f}) "
        f"= {m4:+.4f} [{lo4:+.4f}, {hi4:+.4f}] → {MARK[V['P-4']]}")
run.log(f"      시드별: " + " ".join(f"{x:+.4f}" for x in diff4) + f"  (시드SD {sd4:.4f})")
CROSS = bool(np.mean(wf) < 0 < np.mean(wt))
run.log(f"      부호 교차(전두면<0<횡단면): {'✅ 성립' if CROSS else '❌ 미성립'}"
        + ("  — 같은 전극 수로 소견에 따라 배치가 뒤바뀐다" if CROSS else ""))

# ── 시드 잡음 대비 효과 크기 (실험15d 규약 ②)
def seeds_needed(m, sd, cap=200):
    """t-CI 가 0 을 벗어나려면 시드가 몇 개 필요한가. 없으면 None."""
    if sd <= 0 or m == 0:
        return len(SEEDS)
    for n in range(2, cap + 1):
        if stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n) < abs(m):
            return n
    return None

run.log("\n  【효과 vs 시드 잡음】 실험15d 규약 ② — 효과가 잡음 이하면 표본을 늘려도 안 갈린다")
NEED = {}
for key, nm, mm, ss in (("P-3", "P-3 V1 선택성", m3, sd3), ("P-4", "P-4 교차", m4, sd4)):
    r = abs(mm) / ss if ss > 0 else float("inf")
    n = seeds_needed(mm, ss)
    NEED[key] = n
    if V.get(key) is True:
        tail = "  ✅ 이미 지지 — 추가 시드 불필요"
    elif r < 1:
        tail = "  ⛔ 효과가 잡음 이하 — **검출 불가로 종결**(규약 ②). 시드를 늘리지 않는다"
    elif n is None:
        tail = "  ⛔ 200 시드로도 못 가른다 — 종결"
    else:
        extra = (n - len(SEEDS)) * len(CONFIGS) * K_FOLD
        tail = (f"  ⚠️ 시드 {n}개면 갈린다 (추가 학습 {extra}회 ≈ {extra * 56 / 60:.0f}분) "
                "— 헤드라인이므로 추가 판단 대상")
    run.log(f"      {nm:<14} |효과| {abs(mm):.4f} / 시드SD {ss:.4f} = {r:.2f}배{tail}")

run.log("\n" + "=" * 100)
for k in ("P-1", "P-2", "P-3", "P-4"):
    run.log(f"  {k}: {MARK[V.get(k)]}")
if FATAL:
    run.log("  ⛔ 붕괴 과반 — 위 판정 전부 무효")
run.log("=" * 100)

In [ ]:
# CELL 6 — 그림: 결손 곡선 · V1 이득 · 교차
import matplotlib.pyplot as plt

ORDER = ["II", "I+II", "II+V1", "12"]
COLR = {"전두면": "tab:blue", "횡단면": "tab:red", "혼합": "0.6"}
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))

# (1) 유도를 더할수록 결손이 줄어드는 곡선 — 평면별로 다르게 줄어든다
for s in SITES:
    y = [np.mean(D[c][s]) for c in ORDER]
    e = [t_ci(D[c][s])[3] for c in ORDER]
    ax[0].errorbar(range(4), y, yerr=e, marker="o", capsize=3,
                   color=COLR[SITE_PLANE[s]], alpha=.85,
                   lw=2.2 if SITE_PLANE[s] != "혼합" else 1.0, label=s)
ax[0].axhline(0, color="k", lw=1)
ax[0].axhline(SAT_THR, color="green", ls=":", lw=1)
ax[0].set_xticks(range(4)); ax[0].set_xticklabels(ORDER)
ax[0].set_ylabel("잔여 결손 D = AUROC(12) − AUROC(구성)")
ax[0].set_title("결손 곡선 (오차막대 = 시드 SD)")
ax[0].legend(fontsize=7, ncol=2)

# (2) V1 한 개의 이득 — P-3
xs = np.arange(len(SITES))
gm = [np.mean(G[s]) for s in SITES]
ge = [t_ci(G[s])[3] for s in SITES]
ax[1].bar(xs, gm, yerr=ge, capsize=3, color=[COLR[SITE_PLANE[s]] for s in SITES])
ax[1].axhline(0, color="k", lw=1)
ax[1].set_xticks(xs); ax[1].set_xticklabels(SITES, rotation=45, ha="right")
ax[1].set_ylabel("V1 이득 G = D(II) − D(II+V1)")
ax[1].set_title(f"유도 하나(V1)가 되찾는 양 · P-3 {MARK[V['P-3']]}")

# (3) 교차 — P-4. 전극 2개, 배치만 다르다
wm = [np.mean(W[s]) for s in SITES]
we = [t_ci(W[s])[3] for s in SITES]
ax[2].barh(xs, wm, xerr=we, capsize=3, color=[COLR[SITE_PLANE[s]] for s in SITES])
ax[2].axvline(0, color="k", lw=1)
ax[2].set_yticks(xs); ax[2].set_yticklabels(SITES)
ax[2].set_xlabel("← {I,II} 사지 우세      AUROC 차      가슴 {II,V1} 우세 →")
ax[2].set_title(f"같은 전극 2개, 배치만 다름 · P-4 {MARK[V['P-4']]}")

plt.tight_layout(); run.save_fig("four_configs", fig); plt.show()

In [ ]:
# CELL 7 — 결과 저장
res = {
    "week": 2, "exp_id": "exp16_four_cfg", "quest": "ailab-2026-0015",
    "task": "유도 4구성 확장 — 전극을 하나 더 붙인다면 어느 평면인가",
    "split": "inter",
    "metric": "v1_selectivity_trans_minus_frontal",
    "value": round(float(m3), 4),
    "passed": bool(V.get("P-3") is True and not FATAL),
    "date": time.strftime("%Y-%m-%d"),
    "k_fold": K_FOLD, "n_seeds": len(SEEDS), "configs": list(CONFIGS),
    "sites": SITES, "front": FRONT, "trans": TRANS,
    "alive": ALIVE, "collapse_fatal": FATAL,
    "lead_order_rel_err": lead_chk["rel_err"],
    "drift_vs_exp15_max_abs": (round(max(abs(x) for x in drift.values()), 4)
                               if drift else None),
    "P-1": V.get("P-1"), "P-2": V.get("P-2"), "P-3": V.get("P-3"), "P-4": V.get("P-4"),
    "p3_mean": round(float(m3), 4), "p3_ci": [round(float(lo3), 4), round(float(hi3), 4)],
    "p3_seed_sd": round(float(sd3), 4),
    "p4_mean": round(float(m4), 4), "p4_ci": [round(float(lo4), 4), round(float(hi4), 4)],
    "p4_seed_sd": round(float(sd4), 4), "p4_sign_crossover": CROSS,
    "seeds_needed": NEED,
    "D_mean": {c: {s: round(float(np.mean(D[c][s])), 4) for s in SITES} for c in CONFIGS},
    "auroc_mean": {c: {s: round(float(np.mean(M[c][s]["auroc"])), 4) for s in SITES}
                   for c in CONFIGS},
    "auprc_mean": {c: {s: round(float(np.mean(M[c][s]["auprc"])), 4) for s in SITES}
                   for c in CONFIGS},
    "spec_at_sens90": {c: {s: round(float(np.nanmean(M[c][s]["spec"])), 4) for s in SITES}
                       for c in CONFIGS},
    "alarm_rate": {c: {s: round(float(np.nanmean(M[c][s]["alarm"])), 4) for s in SITES}
                   for c in CONFIGS},
    "v1_gain": {s: round(float(np.mean(G[s])), 4) for s in SITES},
    "crossover_W": {s: round(float(np.mean(W[s])), 4) for s in SITES},
    "verdict": (f"P-1 {MARK[V.get('P-1')]} · P-2 {MARK[V.get('P-2')]} · "
                f"P-3 {MARK[V.get('P-3')]} · P-4 {MARK[V.get('P-4')]}"),
}
res["summary"] = (f"V1 선택성 {m3:+.4f} [{lo3:+.4f},{hi3:+.4f}] · "
                  f"교차 {m4:+.4f} [{lo4:+.4f},{hi4:+.4f}] · "
                  f"부호교차 {'성립' if CROSS else '미성립'} · " + res["verdict"])
run.save_json("result.json", res)
run.log("\n" + json.dumps({k: res[k] for k in
                           ("metric", "value", "passed", "P-1", "P-2", "P-3", "P-4",
                            "p4_sign_crossover", "summary")},
                          ensure_ascii=False, indent=2))
run.finish(res)